<a href="https://colab.research.google.com/github/ArshnoorSinghh/ML-Project/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 4 is best framed as a scoring and ranking task.

I want to give each page an "opportunity score" that measures how far
below the expected click rate for its position it sits. Then I rank the
pages by that score, worst under-earners at the top, so an editor works
down the list.

It's not classification, because "under-earning" isn't a clean yes or no.
A page can be a little below expected or a lot below, so it's a matter of
degree. A score captures that degree; a yes/no label would throw it away.
It's not clustering either, because I'm not discovering unknown groups.
I'm measuring one specific thing (the click gap against a reference) and
ordering pages by it.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no column in the data that says "this page under-earns its
clicks," so I can't use a ready-made target. Instead I build a proxy
from columns I do have.

My proxy is a CTR gap:
- For each position tier, I take the expected CTR as the median CTR of
  pages in that tier.
- For each page, I compare its actual CTR to that expected CTR.
- The gap (expected minus actual) is my target. A large positive gap
  means the page earns far fewer clicks than others at its position,
  so it's a strong under-earner.

This is a proxy, not ground truth. I'm not directly measuring "this page
would benefit from a rewrite," I'm measuring how far below its position
peers it sits, and treating that as a stand-in. Being clear that it's a
proxy matters, because the score is only as good as that assumption.

The specific model (for example a regression or tree-based method) is a
later decision (the modeling assignment). For now I'm only framing the
task, not choosing the algorithm.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My output is a ranked list of pages, worst under-earners first, and an
editor only reviews the top few. So the metric that matters is whether
the top of the list is trustworthy.

I'll use Precision@K: of the top K pages my model flags as under-earning,
what fraction are genuinely under-earning (sitting clearly below the
expected CTR for their position). K is set to how many pages an editor
can review in one batch, for example the top 20 or 50.

I'm not using plain accuracy, because I don't act on all pages, only the
top of the ranked list. A model could look fine on average but still put
weak picks at the top. Precision@K measures exactly the part I use.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [16]:
import pandas as pd
import numpy as np
df= pd.read_csv("/content/content_refresh_anonymized (1).csv")

In [17]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [19]:
df.shape

(30000, 44)

In [20]:
df["expected_ctr"] = df.groupby("position_tier")["ctr"].transform("median")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,expected_ctr
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0.11
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0.03
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0.03
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0.16
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0.03


In [21]:
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,expected_ctr,ctr_gap
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0.11,-0.65
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0.03,-0.02
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0.03,-0.06
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0.16,-0.33
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0.03,-0.10


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag any page below its tier's median CTR" is a
reasonable start, but it assumes position is the only thing that decides
a fair CTR. It isn't. Expected clicks also depend on content type, search
intent, keyword competition, and impression volume, and in w01 I saw
content type alone shift CTR a lot even at the same position.

Two pages at the same position can have genuinely different fair CTRs
because of these other signals. A flat rule can't account for that, and
hand-writing an if-branch for every combination of signals gets brittle
fast. A model can learn how all these features combine to set an expected
CTR, and flag pages that fall below their own fair value, not just the
tier median. That's why this is an ML problem and not a single rule.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.